##1. Buatlah model klasfikasi Multinomial Naive Bayes dengan ketentuan, Menggunakan data spam.csv Fitur CountVectorizer dengan mengaktifkan stop_words Evaluasi hasilnya

In [43]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

d = pd.read_csv('spam.csv', encoding='latin-1').iloc[:, :2]
d.columns = ['Labels', 'SMS']
d['Labels'] = d['Labels'].map({'spam': 1, 'ham': 0})
print(d['Labels'].value_counts())

Xtr, Xte, ytr, yte = train_test_split(d['SMS'].values, d['Labels'].values, test_size=0.2, random_state=50)

bow = CountVectorizer(stop_words='english')
Xtr_bow = bow.fit_transform(Xtr); Xte_bow = bow.transform(Xte)
print('Dimensi train:', Xtr_bow.shape)

mnb_bow = MultinomialNB().fit(Xtr_bow, ytr)
pred_bow = mnb_bow.predict(Xte_bow)
print('Akurasi train:', accuracy_score(ytr, mnb_bow.predict(Xtr_bow)))
print('Akurasi test :', accuracy_score(yte, pred_bow))
print(confusion_matrix(yte, pred_bow))
print(classification_report(yte, pred_bow, target_names=['ham', 'spam'], digits=4))

Labels
0    4825
1     747
Name: count, dtype: int64
Dimensi train: (4457, 7466)
Akurasi train: 0.9946152120260264
Akurasi test : 0.9829596412556054
[[951   3]
 [ 16 145]]
              precision    recall  f1-score   support

         ham     0.9835    0.9969    0.9901       954
        spam     0.9797    0.9006    0.9385       161

    accuracy                         0.9830      1115
   macro avg     0.9816    0.9487    0.9643      1115
weighted avg     0.9829    0.9830    0.9827      1115



Analisis : Model CountVectorizer + stop_words mencapai akurasi test **98.30%** (train 99.46%), dengan 7.466 fitur kata. Dari 161 SMS spam, 145 terdeteksi (recall spam 0.9006), 16 lolos sebagai ham, dan hanya 3 ham salah dianggap spam (precision spam 0.9797). Karena data tidak seimbang (4825 ham : 747 spam), akurasi saja kurang informatif; recall dan F1 untuk kelas spam lebih relevan.

##2. Buatlah model klasfikasi Multinomial Naive Bayes dengan ketentuan, Menggunakan data spam.csv Fitur TF-IDF dengan mengaktifkan stop_words Evaluasi hasilnya dan bandingkan dengan hasil pada Tugas no 2. Berikan kesimpulan fitur mana yang terbaik pada kasus data spam.csv

In [44]:
tfidf = TfidfVectorizer(stop_words='english')
Xtr_tf = tfidf.fit_transform(Xtr); Xte_tf = tfidf.transform(Xte)
print('Dimensi train:', Xtr_tf.shape)

mnb_tf = MultinomialNB().fit(Xtr_tf, ytr)
pred_tf = mnb_tf.predict(Xte_tf)
print('Akurasi train:', accuracy_score(ytr, mnb_tf.predict(Xtr_tf)))
print('Akurasi test :', accuracy_score(yte, pred_tf))
print(confusion_matrix(yte, pred_tf))
print(classification_report(yte, pred_tf, target_names=['ham', 'spam'], digits=4))

# Perbandingan
cmp = pd.DataFrame({
    name: {'Acc train': accuracy_score(ytr, m.predict(Xa)), 'Acc test': accuracy_score(yte, p),
           'Precision spam': precision_score(yte, p), 'Recall spam': recall_score(yte, p), 'F1 spam': f1_score(yte, p)}
    for name, m, Xa, p in [('CountVectorizer', mnb_bow, Xtr_bow, pred_bow), ('TF-IDF', mnb_tf, Xtr_tf, pred_tf)]
}).round(4)
print(cmp)

Dimensi train: (4457, 7466)
Akurasi train: 0.9842943684092439
Akurasi test : 0.9605381165919282
[[954   0]
 [ 44 117]]
              precision    recall  f1-score   support

         ham     0.9559    1.0000    0.9775       954
        spam     1.0000    0.7267    0.8417       161

    accuracy                         0.9605      1115
   macro avg     0.9780    0.8634    0.9096      1115
weighted avg     0.9623    0.9605    0.9579      1115

                CountVectorizer  TF-IDF
Acc train                0.9946  0.9843
Acc test                 0.9830  0.9605
Precision spam           0.9797  1.0000
Recall spam              0.9006  0.7267
F1 spam                  0.9385  0.8417


## Analisis:
TF-IDF memberi akurasi test **96.05%**, lebih rendah dari CountVectorizer (98.30%). Perbedaan utamanya ada di recall spam: TF-IDF hanya menangkap 117 dari 161 spam (recall 0.7267, 44 spam lolos), meski precision-nya sempurna (1.0, tidak ada ham salah diblokir). Artinya model TF-IDF terlalu konservatif. Penyebab yang masuk akal: TF-IDF menormalisasi tiap dokumen (L2) sehingga nilai fitur kecil (<1), dan karena MultinomialNB memakai smoothing alpha=1, smoothing menjadi sangat dominan terhadap nilai fitur kecil tersebut sehingga kelas mayoritas (ham) lebih diuntungkan. Ini penjelasan yang konsisten dengan hasil tetapi tidak saya uji langsung (butuh verifikasi, mis. dengan menurunkan alpha).

##KESIMPULAN :
Pada spam.csv dengan MultinomialNB dan pengaturan default, CountVectorizer (stop_words) adalah fitur terbaik: akurasi 98.30% vs 96.05%, F1 spam 0.9385 vs 0.8417, recall spam 0.90 vs 0.73. TF-IDF hanya unggul di precision (1.0 vs 0.98), tetapi mengorbankan lebih banyak spam yang lolos. Catatan keterbatasan: kesimpulan ini berdasarkan satu split (random_state=50) dan alpha default; TF-IDF mungkin menyusul bila alpha di-tuning, sehingga perbandingan ini adil hanya untuk konfigurasi default sesuai soal.